# Fako Online - Styled Scene Server

**AnimateDiff + ControlNet**

Generates consistent character scenes with style transfer.

### Instructions
1. Set runtime to **T4 GPU**
2. Run all cells in order
3. Copy the ngrok URL to your `.env` file

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
# Cell 2: Load models from Drive
# Account A models
account_a_dir = '/content/drive/MyDrive/AI_Models'

# Check if models exist
animatediff_path = f'{account_a_dir}/AnimateDiff'
sd15_path = f'{account_a_dir}/SD1.5_Base'

if os.path.exists(animatediff_path):
    print(f'AnimateDiff found at {animatediff_path}')
else:
    print(f'WARNING: AnimateDiff not found at {animatediff_path}')

if os.path.exists(sd15_path):
    print(f'SD 1.5 found at {sd15_path}')
else:
    print(f'WARNING: SD 1.5 not found at {sd15_path}')

In [ ]:
# Cell 3: Install dependencies
!pip install -q diffusers transformers accelerate
!pip install -q controlnet-aux
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q imageio[ffmpeg]
print('Dependencies installed!')

In [ ]:
# Cell 4: Import libraries
import torch
import numpy as np
from PIL import Image
from diffusers import AnimateDiffPipeline, ControlNetModel, StableDiffusionControlNetImg2ImgPipeline
from diffusers.utils import export_to_video
import io
import base64

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Cell 5: Load AnimateDiff + ControlNet pipeline
# Load ControlNet for depth conditioning
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11f1p_sd15_depth',
    torch_dtype=torch.float16
)

# Load SD 1.5 base
pipe = AnimateDiffPipeline.from_pretrained(
    f'{account_a_dir}/SD1.5_Base',
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe.to('cuda')

print('AnimateDiff + ControlNet loaded!')

In [ ]:
# Cell 6: Define styled scene generation function
def generate_styled_scene(image_path, prompt, style_prompt, duration=5, num_frames=16):
    """Generate styled scene from image + text prompt."""
    # Load and process input image
    init_image = Image.open(image_path).resize((512, 512))
    
    # Generate video frames
    video_frames = pipe(
        prompt=f'{prompt}, {style_prompt}',
        image=init_image,
        num_frames=num_frames,
        num_inference_steps=20,
        guidance_scale=7.5,
        height=512,
        width=512
    ).frames[0]
    
    output_path = '/content/outputs/styled_scene.mp4'
    export_to_video(video_frames, output_path, fps=16)
    return output_path

print('Styled scene generation function defined!')

In [ ]:
# Cell 7: Define image generation function (SD 1.5 txt2img)
def generate_image(prompt, width=512, height=512):
    """Generate image from text prompt using SD 1.5."""
    from diffusers import StableDiffusionPipeline
    
    sd_pipe = StableDiffusionPipeline.from_pretrained(
        f'{account_a_dir}/SD1.5_Base',
        torch_dtype=torch.float16,
        safety_checker=None
    )
    sd_pipe.to('cuda')
    
    image = sd_pipe(
        prompt=prompt,
        width=width,
        height=height,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    
    output_path = '/content/outputs/generated_image.png'
    image.save(output_path)
    
    # Cleanup
    del sd_pipe
    torch.cuda.empty_cache()
    
    return output_path

print('Image generation function defined!')

In [ ]:
# Cell 8: Create FastAPI server
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title='Fako Online - Styled Scene API')

@app.get('/health')
async def health():
    return {'status': 'ok', 'models': ['animatediff', 'controlnet', 'sd1.5']}

@app.post('/generate-image')
async def api_generate_image(text_prompt: str = Form(...), width: int = Form(512), height: int = Form(512)):
    try:
        image_path = generate_image(text_prompt, width, height)
        return FileResponse(image_path, media_type='image/png')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/generate-styled')
async def api_generate_styled(image: UploadFile = File(...), text_prompt: str = Form(...), style_prompt: str = Form(...), duration: int = Form(5)):
    try:
        image_path = f'/content/outputs/{image.filename}'
        with open(image_path, 'wb') as f:
            f.write(await image.read())
        
        video_path = generate_styled_scene(image_path, text_prompt, style_prompt, duration)
        return FileResponse(video_path, media_type='video/mp4')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

print('FastAPI server defined!')

In [ ]:
# Cell 9: Start server with ngrok
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.bind(8002)
print(f'\nPublic URL: {public_url}')
print(f'\nUpdate your .env file:')
print(f'COLAB_STYLED_URL={public_url}')

uvicorn.run(app, host='0.0.0.0', port=8002)